## Setting up Configuration

In [0]:
%run ./00_setup_config

## Reading data from raw folder

In [0]:
from pyspark.sql.functions import col

try:
    df_raw = spark.read.option("multiline", "true").json(raw_path)
except Exception as e:
    print(f"Error reading JSON: {e}")

df_raw.show(truncate=False)

## Giving Column names + Loading time

In [0]:
from pyspark.sql.functions import col, regexp_extract, current_timestamp

df_flat = df_raw.select(
    col("latitude"),
    col("longitude"),
    col("elevation"),
    col("timezone"),
    col("current.time").alias("weather_time_pkt"),
    col("current.temperature_2m").alias("temperature_c"),
    col("current.relative_humidity_2m").alias("humidity_pct"),
    col("current.apparent_temperature").alias("feels_like_c"),
    col("current.precipitation").alias("precipitation_mm"),
    col("current.weathercode").alias("weather_code"),
    col("current.pressure_msl").alias("pressure_hpa"),
    col("current.wind_speed_10m").alias("windspeed_kmh"),
    col("current.wind_direction_10m").alias("winddirection_deg"),
    col("current.cloud_cover").alias("cloud_cover_pct"),
    col("current.is_day").alias("is_day"),
    col("_metadata.file_path").alias("source_file"),
    regexp_extract(col("_metadata.file_path"), r'([A-Za-z]+)_\d{4}-\d{2}-\d{2}', 1).alias("city"),
    current_timestamp().alias("loading_time")
)

df_flat.show(truncate=False)

## Defining Validation Rules

In [0]:
from pyspark.sql.functions import col, lit, when, concat_ws, array, array_remove

# Define quality checks — each returns a list of failure reasons per row
df_checked = df_flat.withColumn(
    "dq_failures",
    array_remove(array(
        when(col("city").isNull() | (col("city") == ""), "missing_city"),
        when(col("weather_time_pkt").isNull(), "missing_weather_time"),
        when((col("temperature_c") < -30) | (col("temperature_c") > 55), "temperature_out_of_range"),
        when((col("humidity_pct") < 0) | (col("humidity_pct") > 100), "humidity_out_of_range"),
        when(col("windspeed_kmh") < 0, "negative_windspeed"),
        when(col("latitude").isNull() | col("longitude").isNull(), "missing_coordinates")
    ), None)
)

df_checked = df_checked.withColumn("dq_status", when(col("dq_failures").getItem(0).isNotNull(), lit("failed")).otherwise(lit("passed")))

## Spliting into valid & invalid based on validation

In [0]:
df_valid = df_checked.filter(col("dq_status") == "passed").drop("dq_failures", "dq_status")

df_invalid = df_checked.filter(col("dq_status") == "failed") \
    .withColumn("dq_reason", concat_ws(", ", col("dq_failures"))) \
    .drop("dq_failures", "dq_status")

print(f"Passed: {df_valid.count()}, Quarantined: {df_invalid.count()}")

## Putting invalid data in quarantine table

In [0]:
if df_invalid.count() > 0:
    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS internship_databricks_ws.default.weather_quarantine
    USING DELTA
    LOCATION '{processed_path}quarantine/'
    """)
    df_invalid.write.format("delta").mode("append").option("mergeSchema", "true") \
        .saveAsTable("internship_databricks_ws.default.weather_quarantine")
    print(f"Quarantined {df_invalid.count()} rows — see weather_quarantine for reasons")

## Creating table weather_curated pointing to processed folder

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS internship_databricks_ws.default.weather_curated
USING DELTA
LOCATION '{processed_path}weather_curated/'
""")



## Appending valid data to weather_curated

In [0]:
df_valid.write.format("delta").mode("append").option("overwriteSchema", "true").saveAsTable('internship_databricks_ws.default.weather_curated')
print("Appended new run to processed data")

## Dumps old .json files into Done folder

In [0]:

files_to_move = df_raw.select("_metadata.file_path").distinct().collect()

for row in files_to_move:
    source_path = row["file_path"]
    file_name = source_path.split("/")[-1]
    destination_path = f"{raw_path}Done/{file_name}"
    dbutils.fs.mv(source_path, destination_path)
    print(f"Moved: {file_name}")